In [13]:
import os
import matplotlib.pyplot as plt
import numpy as np

from sklearn.datasets import load_files
from pyvi import ViTokenizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import Pipeline

%matplotlib inline

INPUT = 'news_vnexpress'
os.makedirs("images", exist_ok=True)  # Tạo thư mục lưu ảnh sau này

# Thống kê số lượng bài báo theo chủ đề
print('Các nhãn và số văn bản tương ứng trong dữ liệu')
print('------------------------------')
n = 0
for label in os.listdir(INPUT):
    print(f'{label}: {len(os.listdir(os.path.join(INPUT, label)))}')
    n += len(os.listdir(os.path.join(INPUT, label)))
print('------------------------------')
print(f'Tổng số văn bản: {n}')

data_train = load_files(container_path=INPUT, encoding="utf-8")

# Xem mapping giữa nhãn (tên thư mục) và số
print('mapping:')
for i in range(len(data_train.target_names)):
    print(f'{data_train.target_names[i]} - {i}')

print('---------------------')
print(f"File đầu tiên: {data_train.filenames[0]}")
print(f"Nhãn của file đầu tiên: {data_train.target[0]} -> {data_train.target_names[data_train.target[0]]}")
print(f"Nội dung file đầu tiên: {data_train.data[0][:200]}")
print(f"\nTổng số văn bản: {len(data_train.filenames)}")

# Tải danh sách stopwords tiếng Việt
stopwords_file = "vietnamese-stopwords.txt"  # Bạn cần tải file này
with open(stopwords_file, 'r', encoding='utf-8') as f:
    stopwords = f.read().splitlines()

print(f"Số lượng stopwords: {len(stopwords)}")
print(f"Một số stopwords: {stopwords[:10]}")

# Cách 1: Làm thủ công từng bước
vectorizer = CountVectorizer(
    stop_words=stopwords,      # Loại bỏ stopwords
    tokenizer=ViTokenizer.tokenize,  # Tokenizer tiếng Việt
    lowercase=True,            # Chuyển về chữ thường
    max_features=10000         # Chỉ giữ 10000 từ quan trọng nhất
)

# Bước 1: Chuyển văn bản thành ma trận đếm (TF - Term Frequency)
X_counts = vectorizer.fit_transform(data_train.data)

# Bước 2: Chuyển TF thành TF-IDF
tfidf_transformer = TfidfTransformer(norm='l2', use_idf=True)
X_tfidf = tfidf_transformer.fit_transform(X_counts)

# Hoặc dùng Pipeline (gọn hơn)
pipeline = Pipeline([
    ('vect', CountVectorizer(stop_words=stopwords, 
                             tokenizer=ViTokenizer.tokenize,
                             lowercase=True)),
    ('tfidf', TfidfTransformer(norm='l2', use_idf=True))
])

X = pipeline.fit_transform(data_train.data)
Y = data_train.target

print(f"Số lượng từ trong từ điển: {len(pipeline.named_steps['vect'].vocabulary_)}")
print(f"Kích thước dữ liệu sau khi xử lý: {X.shape}")
print(f"Kích thước nhãn tương ứng: {Y.shape}")

# Xem vector TF-IDF của bài báo thứ 101 (index 100)
print(X[100].toarray())
print(f"Nhãn của bài báo này: {Y[100]} - {data_train.target_names[Y[100]]}")

# Đếm số từ khác 0 trong bài báo này
print(f"Số từ có TF-IDF > 0: {sum(X[100].toarray() != 0)}")

# Xem chi tiết các từ và giá trị TF-IDF
feature_names = pipeline.named_steps['vect'].get_feature_names_out()
vector_tfidf = X[100].toarray()[0]
for idx, value in sorted(enumerate(vector_tfidf), key=lambda x: -x[1])[:10]:
    if value > 0:
        print(f"'{feature_names[idx]}': {value:.4f}")
print(X[100])

Các nhãn và số văn bản tương ứng trong dữ liệu
------------------------------
doi-song: 120
du-lich: 54
giai-tri: 201
giao-duc: 105
khoa-hoc: 144
kinh-doanh: 262
phap-luat: 59
suc-khoe: 162
the-thao: 173
thoi-su: 59
------------------------------
Tổng số văn bản: 1339
mapping:
doi-song - 0
du-lich - 1
giai-tri - 2
giao-duc - 3
khoa-hoc - 4
kinh-doanh - 5
phap-luat - 6
suc-khoe - 7
the-thao - 8
thoi-su - 9
---------------------
File đầu tiên: news_vnexpress\khoa-hoc\00133.txt
Nhãn của file đầu tiên: 4 -> khoa-hoc
Nội dung file đầu tiên: Mời độc giả đặt câu hỏi tại đây


Tổng số văn bản: 1339
Số lượng stopwords: 1942
Một số stopwords: ['a lô', 'a ha', 'ai', 'ai ai', 'ai nấy', 'ai đó', 'alô', 'amen', 'anh', 'anh ấy']
Số lượng từ trong từ điển: 133
Kích thước dữ liệu sau khi xử lý: (1339, 133)
Kích thước nhãn tương ứng: (1339,)
[[0.00076531 0.73775912 0.         0.01336405 0.         0.0106016
  0.         0.         0.00407259 0.00406941 0.         0.
  0.04891094 0.00317183 0.03274077 0.